# CVRP Optimization: Exact Method (ILP)

## Mathematical Formulation

* **Decision Variables:** * $x_{ij} \in \{0, 1\}$: $1$ if a vehicle travels from node $i$ to $j$.
  * $u_i$: Auxiliary variable for capacity/sub-tour tracking.
* **Objective:**
  * Minimize $Z = \sum_{i} \sum_{j} c_{ij} x_{ij}$ (Total distance).
* **Constraints:**
  1. Every customer must be visited exactly once.
  2. Vehicles must return to the depot.
  3. **Capacity Constraint:** Total demand on a route $\le$ Vehicle Capacity $Q$.


## Data Parsing Component

In [2]:
!pip install scipy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.3/30.3 MB 242.6 kB/s  0:02:05m0:00:0100:04

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip


In [4]:
!pip install pulp

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 241.9 kB/s  0:01:08m0:00:0100:03

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip


In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import distance_matrix
import pulp

def load_vrp_data(filepath):
    """Parses the .vrp file to extract nodes, demands, and capacity."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    capacity = int([l for l in lines if "CAPACITY" in l][0].split(":")[1])
    
    # Find start of sections
    coord_start = lines.index("NODE_COORD_SECTION\t\t\n") + 1
    demand_start = lines.index("DEMAND_SECTION\t\t\n") + 1
    dimension = int([l for l in lines if "DIMENSION" in l][0].split(":")[1])

    # Parse Coordinates
    coords = []
    for i in range(coord_start, coord_start + dimension):
        parts = lines[i].split()
        coords.append((float(parts[1]), float(parts[2])))

    # Parse Demands
    demands = []
    for i in range(demand_start, demand_start + dimension):
        demands.append(int(lines[i].split()[1]))

    return np.array(coords), demands, capacity

# Load your specific Kaggle files
all_coords, all_demands, capacity = load_vrp_data('data/X-n101-k25.vrp')

# Select the subset (First 10 nodes for the Exact Method test)
# Always keep the first node (the Depot) and then take a subset of customers
N_SUBSET = 10

# Extract the depot (index 0)
depot_coords = all_coords[0:1]
depot_demand = all_demands[0:1]

# Extract the customers (indices 1 to N_SUBSET)
customer_coords = all_coords[1:N_SUBSET+1]
customer_demands = all_demands[1:N_SUBSET+1]

# Combine them
coords = np.vstack((depot_coords, customer_coords))
demands = depot_demand + customer_demands
N = len(coords)
dist_mat = distance_matrix(coords, coords)

## ILP Model Implementation

In [22]:
# 1. Initialize Problem
prob = pulp.LpProblem("CVRP_Exact_ILP", pulp.LpMinimize)

# 2. Decision Variables
x = pulp.LpVariable.dicts("x", [(i, j) for i in range(N) for j in range(N)], cat='Binary')
u = pulp.LpVariable.dicts("u", range(N), lowBound=0, upBound=capacity)

# 3. Objective Function: Minimize Total Distance
prob += pulp.lpSum(dist_mat[i][j] * x[i, j] for i in range(N) for j in range(N) if i != j)

# 4. Constraints
for i in range(N):
    # Every customer visited once
    prob += pulp.lpSum(x[j, i] for j in range(N) if i != j) == 1
    prob += pulp.lpSum(x[i, j] for j in range(N) if i != j) == 1

# Sub-tour elimination & Capacity (MTZ Formulation)
for i in range(1, N):
    for j in range(1, N):
        if i != j:
            prob += u[i] - u[j] + capacity * x[i, j] <= capacity - demands[j]

for i in range(1, N):
    prob += u[i] >= demands[i]

# 5. Solver with 2-minute time limit (as the problem is NP-Hard)
solver = pulp.PULP_CBC_CMD(timeLimit=300, gapRel=0.05, msg=1)
prob.solve(solver)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/dxdy/Library/Python/3.9/lib/python/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/ml/czjfmszj3r3125pzm_gw9mdc0000gn/T/7e986a3387ae430c935309f072369298-pulp.mps -sec 300 -ratio 0.05 -timeMode elapsed -branch -printingOptions all -solution /var/folders/ml/czjfmszj3r3125pzm_gw9mdc0000gn/T/7e986a3387ae430c935309f072369298-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 127 COLUMNS
At line 958 RHS
At line 1081 BOUNDS
At line 1202 ENDATA
Problem MODEL has 122 rows, 120 columns and 500 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 300
ratioGap was changed from 0 to 0.05
Option for timeMode changed from cpu to elapsed
Continuous objective value is 2205.78 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 90 strengthened rows, 0 substitutions
Cgl0003I 0 fixed, 0 tightened bounds, 90 strengthened rows, 0 substit

-1

In [20]:
import matplotlib.pyplot as plt

def plot_cvrp_results(coords, x_vars, title="ILP Optimized Routes"):
    plt.figure(figsize=(8, 8))
    n = len(coords)
    for i in range(n):
        for j in range(n):
            if pulp.value(x_vars[i, j]) > 0.9:
                plt.arrow(coords[i, 0], coords[i, 1], 
                          coords[j, 0] - coords[i, 0], coords[j, 1] - coords[i, 1], 
                          head_width=5, color='blue', alpha=0.6, length_includes_head=True)

    plt.scatter(coords[1:, 0], coords[1:, 1], c='red', label='Customers')
    plt.scatter(coords[0, 0], coords[0, 1], c='green', marker='s', s=100, label='Depot')
    plt.title(title)
    plt.legend()
    plt.show()

if prob.status == 1: # 1 means 'Optimal'
    plot_cvrp_results(coords, x, title=f"ILP Route for {N_SUBSET} Nodes")